# Arkenstone Discovery V6 — multi-experiment GPU campaign

Runs ARK-011 → ARK-012 → ARK-013 → ARK-014 under a 240-minute budget. The notebook pins an audited commit, compiles all runners, executes a GPU smoke test, then runs the campaign. Results are packaged even on failure.


In [ ]:
import os, shutil, subprocess, sys, torch
PINNED_RUNNER_COMMIT = '2238f5519c3225f44010e74ba6d1af9f31d8524f'
REPO = '/content/An-Ra-the-new-AGI-discovery-v6'
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','30','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
paths = [
 'experiments/COLAB/discovery_v6_common.py',
 'experiments/COLAB/run_discovery_v6.py',
 'experiments/ARK-011/run_ark011.py',
 'experiments/ARK-012/run_ark012.py',
 'experiments/ARK-013/run_ark013.py',
 'experiments/ARK-014/run_ark014.py',
 'experiments/ARK-001/run_ark001.py',
 'experiments/lib/ark_tasks.py',
]
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
runner = os.path.join(REPO,'experiments/COLAB/run_discovery_v6.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
subprocess.run([sys.executable, runner, '--smoke-test', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env, check=True)
print('\nGPU SMOKE TEST PASS — safe to start the full multi-experiment campaign')


In [ ]:
# Full campaign: no filler/sleeping; up to four preregistered experiments share a 240-minute budget.
import os, subprocess, sys
runner = os.path.join(REPO,'experiments/COLAB/run_discovery_v6.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--budget-minutes', '240', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env)
print('FULL CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('The runner wrote a failure receipt and packaged completed partial JSONs. Do not rerun before inspecting the ZIP.')


In [ ]:
from pathlib import Path
root = Path('/content/arkenstone_discovery_v6_results')
print('Result files:')
for p in sorted(root.glob('*')):
    print(p.name, p.stat().st_size if p.is_file() else '')
zip_path = root / 'ARKENSTONE_DISCOVERY_V6_RESULTS.zip'
if zip_path.exists():
    print('\nZIP:', zip_path, zip_path.stat().st_size, 'bytes')
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print('Manual download: left Files panel → content → arkenstone_discovery_v6_results → ZIP', exc)
